# When might supply fall short?

Compare the timing of capacity and demand in a ten-year experiment, then vary the composition of end use. Edit the parameter cells to explore other paths. [Model assumptions](../docs/methodology.md)

In [1]:
from pathlib import Path
import sys

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/oil_evidence/models.py").is_file())
sys.path.insert(0, str(root / "src"))
data = root / "data"
import csv
from itertools import product
from oil_evidence.models import capacity_path, first_shortfall, sector_mix

## How quickly does spare capacity disappear?

Opening capacity is 105 against demand of 100. All units measure flow relative to opening demand. A delay postpones the first delivery of new capacity. The model holds intended demand fixed along each chosen growth path.

In [2]:
decline = 0.04
additions = 3
for growth, delay in [(0, 0), (0, 2), (-0.02, 0)]:
    path = list(capacity_path(decline, additions, growth, delay))
    cross = first_shortfall(path)
    label = f"year {cross}" if cross is not None else "none within 10 years"
    print(f"Demand growth {growth:>5.0%}, delay {delay}: first shortfall {label}, year-10 margin {path[-1]['margin']:+.2f}")

Demand growth    0%, delay 0: first shortfall year 5, year-10 margin -5.06
Demand growth    0%, delay 2: first shortfall year 2, year-10 margin -9.30
Demand growth   -2%, delay 0: first shortfall none within 10 years, year-10 margin +13.24


![Three illustrative capacity margins](../outputs/capacity-margin.png)

The figure shows the published parameter choices. The cell above recalculates the margin for your edits.

In [3]:
grid = list(product((0.02, 0.04, 0.06), (1, 3, 5), (-0.02, 0, 0.01), (0, 2)))
crossings = sum(first_shortfall(capacity_path(*settings)) is not None for settings in grid)
print(f"{crossings} of {len(grid)} chosen paths cross within ten years.")

35 of 54 chosen paths cross within ten years.


The count depends on which inputs are included. The grid has no probability weights. A negative margin indicates the size of the adjustment required before prices, inventories, suppliers or buyers respond.

## Could a shrinking market still run short?

Hold demand growth at −2% a year and capacity decline at 4%. Compare two delivery schedules from year one. Both begin with five spare index points and projects start on time. Only the amount of new capacity delivered each year changes. These are two rows of the [published grid](../outputs/sensitivity.csv). The investment decision that would lead to either schedule is outside the model.

In [4]:
for new_capacity in (3, 1):
    path = list(capacity_path(decline=0.04, additions=new_capacity, growth=-0.02))
    year = first_shortfall(path)
    crossing = f"year {year}" if year is not None else "none within 10 years"
    print(f"{new_capacity} added per year: first shortfall {crossing}, "
          f"year-10 margin {path[-1]['margin']:+.2f} index points")

3 added per year: first shortfall none within 10 years, year-10 margin +13.24 index points
1 added per year: first shortfall year 5, year-10 margin -3.52 index points


## What if road transport uses less oil?

Start with the [2024 OPEC end-use baseline](../data/README.md). Hold other uses constant. Vary the reduction in road use and the change in petrochemical demand independently.

In [5]:
with (data / "sector_baseline.csv").open(newline="") as stream:
    base, = csv.DictReader(stream)
levels = [float(base[key]) for key in ("world_mbd", "road_mbd", "petrochemicals_mbd")]
for road_cut, chemical_growth in [(0, 0), (0.5, 0), (0.5, -0.2)]:
    result = sector_mix(*levels, road_cut, chemical_growth)
    print(f"Road cut {road_cut:.0%}, chemicals {chemical_growth:+.0%}: "
          f"world {result['total_mbd']:.1f} mb/d, petrochemicals {result['chemicals_mbd']:.1f} mb/d "
          f"({result['chemicals_share_pct']:.2f}% of demand)")

Road cut 0%, chemicals +0%: world 103.7 mb/d, petrochemicals 15.5 mb/d (14.95% of demand)
Road cut 50%, chemicals +0%: world 80.7 mb/d, petrochemicals 15.5 mb/d (19.21% of demand)
Road cut 50%, chemicals -20%: world 77.6 mb/d, petrochemicals 12.4 mb/d (15.98% of demand)


In the final case the petrochemical share rises even as its volume falls. This composition effect says nothing by itself about a particular product’s dependence on crude or a buyer’s ability to afford a substitute.